# 01 · Kiểm tra dữ liệu (Data Audit)

Mục tiêu: **tin được dữ liệu trước khi huấn luyện**. Notebook này kiểm tra:

1. **Tính toàn vẹn file**: ảnh thiếu nhãn, mask mồ côi, file hỏng, kích thước ảnh ≠ kích thước mask.
2. **Chất lượng nhãn**: box suy biến / vượt biên / quá nhỏ, instance bị lọc vì diện tích nhỏ, instance bị vỡ thành nhiều mảnh.
3. **Phân bố số quả mỗi ảnh** (biến mục tiêu của bài toán đếm) và độ cân bằng giữa train / val / test.
4. **Kích thước quả, vị trí, mức độ chồng lấn** — quyết định chọn `imgsz`, tiling, `sigma` của density map.
5. **Ảnh gần trùng lặp** giữa các split (rò rỉ dữ liệu).

> Chạy được cả local và Kaggle. Nếu chưa có dữ liệu thật, notebook tự sinh **dữ liệu giả** để chạy thử.

In [ ]:
import os, sys, json, time, shutil, warnings
from pathlib import Path

# ---- Xác định thư mục gốc project: chạy được cả local lẫn Kaggle ----
IN_KAGGLE = Path('/kaggle/working').exists()
if IN_KAGGLE:
    CODE_INPUT = Path('/kaggle/input/fruitcount-code/FruitCount')   # sửa theo tên Kaggle Dataset chứa code
    ROOT = Path('/kaggle/working/FruitCount')
    if not ROOT.exists():
        shutil.copytree(CODE_INPUT, ROOT)
    OUT_BASE = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve()
    while ROOT != ROOT.parent and not (ROOT / 'src').is_dir():
        ROOT = ROOT.parent
    OUT_BASE = ROOT / 'outputs'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image as IPImage

from src.utils import load_yaml, save_yaml, read_json, write_json, load_image, ensure_dir

DS_CFG_PATH = Path(os.environ.get('FRUITCOUNT_DATASET_CFG', ROOT / 'configs/dataset.yaml'))
DS_CFG = load_yaml(DS_CFG_PATH)
PROCESSED = ROOT / DS_CFG.get('processed_dir', 'data/processed')
RUNS = Path(os.environ.get('FRUITCOUNT_RUNS', OUT_BASE / 'runs' if IN_KAGGLE else ROOT / 'runs'))
OUT_DIR = ensure_dir(OUT_BASE / 'notebooks' / '01_data_audit')   # hình + bảng của notebook này

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3,
                     'axes.spines.top': False, 'axes.spines.right': False})
FAMILY_COLORS = {'detection': '#4C72B0', 'segmentation': '#55A868', 'density': '#C44E52'}


def savefig(fig, name):
    fig.savefig(OUT_DIR / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


class StopExecution(Exception):
    """Dừng 'Run All' một cách êm (không in traceback)."""
    def _render_traceback_(self):
        return []


try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

print(f'Môi trường : {"Kaggle" if IN_KAGGLE else "Local"}')
print(f'ROOT       : {ROOT}')
print(f'PROCESSED  : {PROCESSED}')
print(f'RUNS       : {RUNS}')
print(f'OUT_DIR    : {OUT_DIR}')

## 0. Tham số

In [ ]:
USE_SYNTHETIC_IF_MISSING = True   # chưa có dữ liệu thật -> sinh dữ liệu giả để chạy thử
FORCE_RELOAD = False              # True = đọc lại annotation gốc thay vì records.json
SCAN_MASKS = True                 # quét lại mask: instance nhỏ bị lọc, instance bị vỡ mảnh
MAX_SCAN = None                   # giới hạn số ảnh khi quét mask (None = tất cả)
DUP_HAMMING = 4                   # ngưỡng khoảng cách Hamming (dHash 64 bit) coi là "gần trùng"
CROWD_IOU = 0.1                   # box có IoU > ngưỡng này với box khác = "chồng lấn"
YOLO_IMGSZ = load_yaml(ROOT / 'configs/models/yolov8.yaml')['model']['imgsz']
SPLITS = ['train', 'val', 'test']

## 1. Nạp dữ liệu

In [ ]:
import cv2

def find_dataset_dirs(base='/kaggle/input'):
    """Tìm cặp thư mục images/ + masks/ (ưu tiên đường dẫn có chữ 'train')."""
    base = Path(base)
    if not base.exists():
        return None, None
    cands = [d for d in base.rglob('images') if d.is_dir() and (d.parent / 'masks').is_dir()]
    cands.sort(key=lambda d: ('train' not in str(d).lower(), len(str(d))))
    return (cands[0], cands[0].parent / 'masks') if cands else (None, None)


def make_synthetic_dataset(root, n=60, size=(480, 640), seed=0):
    """Ảnh 'vườn' giả: quả tròn đỏ trên nền xanh, CÓ chồng lấn (quả sau đè quả trước) + instance mask."""
    rng = np.random.default_rng(seed)
    img_dir, mask_dir = ensure_dir(Path(root) / 'images'), ensure_dir(Path(root) / 'masks')
    h, w = size
    for k in range(n):
        img = np.full((h, w, 3), (40, 110, 40), np.float32) + rng.normal(0, 12, (h, w, 3))
        img = np.clip(img, 0, 255).astype(np.uint8)
        mask = np.zeros((h, w), np.uint16)
        for i in range(1, int(rng.integers(3, 70)) + 1):
            r = int(rng.integers(6, 18))
            cx, cy = int(rng.integers(r, w - r)), int(rng.integers(r, h - r))
            color = tuple(int(c) for c in (rng.integers(170, 235), rng.integers(20, 80), rng.integers(20, 60)))
            cv2.circle(img, (cx, cy), r, color, -1)
            cv2.circle(mask, (cx, cy), r, i, -1)
        cv2.imwrite(str(img_dir / f'syn_{k:03d}.png'), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        cv2.imwrite(str(mask_dir / f'syn_{k:03d}.png'), mask)


paths = DS_CFG['paths']
if IN_KAGGLE and not Path(paths['images_dir']).exists():
    found_img, found_mask = find_dataset_dirs()
    if found_img is not None:
        paths.update(images_dir=str(found_img), masks_dir=str(found_mask))
        save_yaml(DS_CFG, DS_CFG_PATH)
        print('Tự phát hiện dữ liệu trên Kaggle:', found_img)

SYNTHETIC = not Path(paths['images_dir']).exists()
if SYNTHETIC:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(f"Không thấy {paths['images_dir']} — sửa `paths` trong {DS_CFG_PATH}")
    syn_root = OUT_BASE / 'synthetic_data'
    if not (syn_root / 'images').exists():
        make_synthetic_dataset(syn_root)
    DS_CFG = {**DS_CFG, 'name': 'synthetic', 'format': 'instance_masks',
              'paths': {**paths, 'images_dir': str(syn_root / 'images'),
                        'masks_dir': str(syn_root / 'masks'), 'mask_suffix': ''}}
    paths = DS_CFG['paths']
    PROCESSED = syn_root / 'processed'
    display(Markdown('> ⚠️ **Không tìm thấy dữ liệu thật** → đang dùng **dữ liệu giả** để chạy thử notebook. '
                     'Sửa `paths` trong `configs/dataset.yaml` rồi chạy lại để có số liệu thật.'))

from src.data import load_records, load_records_from_config, load_splits, make_splits, save_records, save_splits


def load_records_and_splits(force=False):
    """Ưu tiên records.json/splits.json (đã chạy prepare_data.py); nếu chưa có thì đọc annotation gốc."""
    rp, sp = PROCESSED / 'records.json', PROCESSED / 'splits.json'
    if rp.exists() and sp.exists() and not force:
        return load_records(rp), load_splits(sp), 'records.json + splits.json'
    t0 = time.time()
    recs = load_records_from_config(DS_CFG)
    c = DS_CFG.get('split', {}) or {}
    spl = make_splits(recs, val=c.get('val', 0.15), test=c.get('test', 0.15), seed=c.get('seed', 42),
                      stratify_by_count=c.get('stratify_by_count', True))
    if SYNTHETIC:   # chỉ ghi cho dữ liệu giả; dữ liệu thật hãy chạy scripts/prepare_data.py
        save_records(recs, rp)
        save_splits(spl, sp)
    return recs, spl, f'annotation gốc ({time.time() - t0:.1f}s)'

In [ ]:
records, splits, source = load_records_and_splits(force=FORCE_RELOAD)
id2split = {i: s for s, ids in splits.items() for i in ids}
rec_by_id = {r.image_id: r for r in records}

print(f"Dataset   : {DS_CFG.get('name')}  (format = {DS_CFG.get('format')})")
print(f'Nguồn     : {source}')
print(f'Số ảnh    : {len(records)} | tổng số quả: {sum(r.count for r in records)}')
print('Split     :', {k: len(v) for k, v in splits.items()})

## 2. Toàn vẹn file
Ảnh không có nhãn sẽ bị **bỏ qua im lặng** khi đọc annotation → cần biết có bao nhiêu ảnh bị mất.

In [ ]:
from collections import Counter
from src.data.annotations import list_images, IMAGE_EXTS

img_dir = Path(paths['images_dir'])
images = list_images(img_dir) if img_dir.exists() else []
img_stems = {p.stem for p in images}
rec_ids = {r.image_id for r in records}

issues = {
    'Ảnh trong thư mục nhưng không có nhãn': sorted(img_stems - rec_ids),
    'image_id bị trùng': [i for i, c in Counter(r.image_id for r in records).items() if c > 1],
    'File ảnh của record không tồn tại': [r.image_id for r in records if not Path(r.image_path).exists()],
    'Ảnh có 0 quả': [r.image_id for r in records if r.count == 0],
}
if DS_CFG.get('format') == 'instance_masks' and paths.get('masks_dir') and Path(paths['masks_dir']).exists():
    suf = paths.get('mask_suffix') or ''
    mask_stems = {p.stem[:-len(suf)] if suf and p.stem.endswith(suf) else p.stem
                  for p in Path(paths['masks_dir']).iterdir() if p.suffix.lower() in IMAGE_EXTS}
    issues['Mask không có ảnh tương ứng'] = sorted(mask_stems - img_stems)

df_issues = pd.DataFrame([{'kiểm tra': k, 'số lượng': len(v), 'ví dụ': ', '.join(map(str, v[:5]))}
                          for k, v in issues.items()])
display(df_issues)

In [ ]:
# Kích thước thật của ảnh vs kích thước trong annotation (với instance mask: lấy từ mask)
from PIL import Image

rows = []
for r in tqdm(records, desc='đọc header ảnh'):
    try:
        with Image.open(r.image_path) as im:
            w, h = im.size
    except Exception:
        w = h = -1
    rows.append((r.image_id, r.width, r.height, w, h))
df_size = pd.DataFrame(rows, columns=['image_id', 'ann_w', 'ann_h', 'img_w', 'img_h'])
unreadable = df_size[df_size.img_w < 0]
mismatch = df_size[(df_size.img_w >= 0) & ((df_size.ann_w != df_size.img_w) | (df_size.ann_h != df_size.img_h))]
issues['Ảnh không đọc được'] = unreadable.image_id.tolist()
issues['Kích thước ảnh ≠ annotation'] = mismatch.image_id.tolist()
print(f'Ảnh không đọc được : {len(unreadable)}')
print(f'Lệch kích thước    : {len(mismatch)}')
if len(mismatch):
    display(mismatch.head(10))
display(df_size.groupby(['img_w', 'img_h']).size().rename('số ảnh').reset_index()
        .sort_values('số ảnh', ascending=False).head(10))

## 3. Quét instance mask
- `min_instance_area` trong `dataset.yaml` loại bỏ instance quá nhỏ (nhiễu). Cần biết **bao nhiêu quả thật bị loại**.
- Một id mask gồm nhiều mảnh rời nhau thường là **quả bị lá che cắt đôi** (vẫn là 1 quả) hoặc **lỗi gán nhãn** (2 quả chung id).

In [ ]:
mask_scan = None
if SCAN_MASKS and DS_CFG.get('format') == 'instance_masks':
    from src.data.annotations import read_instance_mask
    min_area = DS_CFG.get('min_instance_area', 20)
    rows, all_areas = [], []
    for r in tqdm(records[:MAX_SCAN], desc='quét mask'):
        if not r.mask_path:
            continue
        inst = read_instance_mask(r.mask_path)
        ids, areas = np.unique(inst[inst > 0], return_counts=True)
        all_areas.append(areas)
        n_frag = 0
        for (x1, y1, x2, y2), i in zip(r.boxes.astype(int), r.instance_ids):
            sub = (inst[y1:y2, x1:x2] == i).astype(np.uint8)
            n_cc, _ = cv2.connectedComponents(sub)
            n_frag += (n_cc - 1) > 1
        rows.append({'image_id': r.image_id, 'n_ids_raw': len(ids), 'n_kept': r.count,
                     'n_dropped_small': int((areas < min_area).sum()), 'n_fragmented': int(n_frag)})
    mask_scan = pd.DataFrame(rows)
    all_areas = np.concatenate(all_areas) if all_areas else np.zeros(0)

    print(f'Instance gốc      : {mask_scan.n_ids_raw.sum()}')
    print(f'Bị lọc (< {min_area}px) : {mask_scan.n_dropped_small.sum()} '
          f'({mask_scan.n_dropped_small.sum() / max(1, mask_scan.n_ids_raw.sum()):.2%})')
    print(f'Instance vỡ mảnh  : {mask_scan.n_fragmented.sum()} '
          f'({mask_scan.n_fragmented.sum() / max(1, mask_scan.n_kept.sum()):.2%}) ở {(mask_scan.n_fragmented > 0).sum()} ảnh')

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(np.log10(np.maximum(all_areas, 1)), bins=60, color='#4C72B0', edgecolor='white')
    ax.axvline(np.log10(min_area), color='red', ls='--', label=f'min_instance_area = {min_area}px')
    ax.set(xlabel='log10(diện tích instance, px)', ylabel='số instance', title='Phân bố diện tích instance trong mask')
    ax.legend()
    savefig(fig, 'mask_instance_area')
    display(mask_scan.sort_values('n_fragmented', ascending=False).head(5))
else:
    print('Bỏ qua (định dạng không phải instance_masks hoặc SCAN_MASKS=False).')

## 4. Bảng thống kê theo ảnh và theo box
**Chồng lấn**: với mỗi box tính IoU lớn nhất với box khác cùng ảnh, và tỉ lệ diện tích bị box khác phủ (IoA).
Đây là đại lượng xấp xỉ mức độ quả chen chúc / che khuất — chính là biến số của câu hỏi nghiên cứu.

In [ ]:
def pairwise_overlap(b):
    """Trả về (max IoU với box khác, max IoA = tỉ lệ diện tích box bị box khác phủ)."""
    n = len(b)
    if n < 2:
        return np.zeros(n), np.zeros(n)
    x1 = np.maximum(b[:, None, 0], b[None, :, 0]); y1 = np.maximum(b[:, None, 1], b[None, :, 1])
    x2 = np.minimum(b[:, None, 2], b[None, :, 2]); y2 = np.minimum(b[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    iou = inter / (area[:, None] + area[None, :] - inter + 1e-9)
    ioa = inter / (area[:, None] + 1e-9)
    np.fill_diagonal(iou, 0); np.fill_diagonal(ioa, 0)
    return iou.max(1), ioa.max(1)


img_rows, box_rows = [], []
for r in records:
    b = r.boxes.astype(np.float64)
    W, H = r.width, r.height
    bw, bh = b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]
    side = np.sqrt(np.clip(bw * bh, 0, None))
    max_iou, max_ioa = pairwise_overlap(b)
    border = (b[:, 0] <= 1) | (b[:, 1] <= 1) | (b[:, 2] >= W - 1) | (b[:, 3] >= H - 1)
    yolo_scale = YOLO_IMGSZ / max(W, H)
    img_rows.append({
        'image_id': r.image_id, 'split': id2split.get(r.image_id, '?'), 'width': W, 'height': H,
        'count': r.count, 'mean_side': side.mean() if len(b) else np.nan,
        'median_side': np.median(side) if len(b) else np.nan,
        'crowded_frac': (max_iou > CROWD_IOU).mean() if len(b) else 0.0,
        'mean_max_iou': max_iou.mean() if len(b) else 0.0,
        'fruits_per_mpx': r.count / (W * H / 1e6), 'border_frac': border.mean() if len(b) else 0.0,
    })
    for k in range(len(b)):
        box_rows.append({
            'image_id': r.image_id, 'split': id2split.get(r.image_id, '?'), 'img_w': W, 'img_h': H,
            'x1': b[k, 0], 'y1': b[k, 1], 'x2': b[k, 2], 'y2': b[k, 3], 'w': bw[k], 'h': bh[k],
            'side': side[k], 'side_yolo': side[k] * yolo_scale, 'aspect': bw[k] / max(bh[k], 1e-6),
            'cx_n': (b[k, 0] + b[k, 2]) / 2 / W, 'cy_n': (b[k, 1] + b[k, 3]) / 2 / H,
            'max_iou': max_iou[k], 'max_ioa': max_ioa[k], 'border': bool(border[k]),
        })

df_img = pd.DataFrame(img_rows)
df_box = pd.DataFrame(box_rows)
print(f'df_img: {df_img.shape} | df_box: {df_box.shape}')
display(df_img.describe().T.round(3))

## 5. Kiểm tra hợp lệ của box

In [ ]:
checks = {
    'w ≤ 0 hoặc h ≤ 0': (df_box.w <= 0) | (df_box.h <= 0),
    'Vượt biên ảnh': (df_box.x1 < -0.5) | (df_box.y1 < -0.5) | (df_box.x2 > df_box.img_w + 0.5) | (df_box.y2 > df_box.img_h + 0.5),
    'Cạnh ngắn < 4 px': np.minimum(df_box.w, df_box.h) < 4,
    'Tỉ lệ khung > 3 hoặc < 1/3': (df_box.aspect > 3) | (df_box.aspect < 1 / 3),
    'Gần trùng box khác (IoU > 0.9)': df_box.max_iou > 0.9,
    'Chạm mép ảnh (quả bị cắt)': df_box.border,
}
df_checks = pd.DataFrame([{'kiểm tra': k, 'số box': int(m.sum()), 'tỉ lệ': f'{m.mean():.2%}',
                           'số ảnh': df_box.loc[m, 'image_id'].nunique()} for k, m in checks.items()])
display(df_checks)

## 6. Phân bố số quả mỗi ảnh
Biến mục tiêu của bài toán. Các nhóm (bin) dưới đây **trùng với nhóm dùng trong `error_by_count_bin`** khi đánh giá,
nên bảng này cho biết mỗi nhóm có bao nhiêu ảnh test — nhóm quá ít ảnh thì MAE theo nhóm sẽ nhiễu.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
bins = np.histogram_bin_edges(df_img['count'], bins=min(40, max(10, df_img['count'].nunique())))
axes[0].hist(df_img['count'], bins=bins, color='#4C72B0', edgecolor='white')
axes[0].axvline(df_img['count'].mean(), color='red', ls='--', label=f"TB = {df_img['count'].mean():.1f}")
axes[0].axvline(df_img['count'].median(), color='orange', ls=':', label=f"trung vị = {df_img['count'].median():.0f}")
axes[0].set(title='Số quả mỗi ảnh (toàn bộ)', xlabel='số quả', ylabel='số ảnh'); axes[0].legend()
for s in SPLITS:
    v = df_img.loc[df_img.split == s, 'count']
    axes[1].hist(v, bins=bins, density=True, histtype='step', lw=2, label=f'{s} (n={len(v)})')
    x = np.sort(v); axes[2].plot(x, np.arange(1, len(x) + 1) / max(1, len(x)), lw=2, label=s)
axes[1].set(title='Theo split (chuẩn hóa)', xlabel='số quả'); axes[1].legend()
axes[2].set(title='Hàm phân phối tích lũy (ECDF)', xlabel='số quả', ylabel='tỉ lệ ảnh'); axes[2].legend()
savefig(fig, 'count_distribution')

display(df_img.groupby('split')['count'].agg(['count', 'sum', 'mean', 'std', 'min', 'median', 'max'])
        .rename(columns={'count': 'n_images', 'sum': 'total_fruits'}).reindex(SPLITS).round(2))

In [ ]:
from src.evaluation.error_analysis import DEFAULT_BINS

bin_labels = [f'{lo}-{hi - 1}' if hi < 10**8 else f'>={lo}' for lo, hi in zip(DEFAULT_BINS[:-1], DEFAULT_BINS[1:])]
df_img['count_bin'] = pd.cut(df_img['count'], bins=list(DEFAULT_BINS), right=False, labels=bin_labels)
display(pd.crosstab(df_img['count_bin'], df_img['split'], margins=True, margins_name='tổng')
        .reindex(columns=SPLITS + ['tổng']))

try:
    from scipy.stats import ks_2samp
    rows = []
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        st = ks_2samp(df_img.loc[df_img.split == a, 'count'], df_img.loc[df_img.split == b, 'count'])
        rows.append({'cặp': f'{a} vs {b}', 'KS stat': round(st.statistic, 3), 'p-value': round(st.pvalue, 3)})
    print('Kiểm định Kolmogorov–Smirnov (p lớn = phân bố số quả giữa hai split không khác biệt):')
    display(pd.DataFrame(rows))
except ImportError:
    print('Chưa cài scipy — bỏ qua kiểm định KS.')

## 7. Kích thước quả
- `side = sqrt(w·h)` tính theo pixel ảnh gốc; `side_yolo` là kích thước sau khi YOLO resize cạnh dài về `imgsz`.
- Ngưỡng COCO: *small* < 32 px, *medium* 32–96 px. Quả nhỏ sau resize là dấu hiệu nên **tăng `imgsz`** hoặc **bật tiling**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
lb = np.logspace(np.log10(max(1, df_box.side.min())), np.log10(df_box.side.max() + 1), 50)
axes[0].hist(df_box.side, bins=lb, alpha=0.7, label='ảnh gốc')
axes[0].hist(df_box.side_yolo, bins=lb, alpha=0.7, label=f'sau resize imgsz={YOLO_IMGSZ}')
for t in (32, 96):
    axes[0].axvline(t, color='k', ls='--', lw=1)
axes[0].set_xscale('log'); axes[0].set(title='Kích thước quả sqrt(w·h)', xlabel='px'); axes[0].legend()
axes[1].hist(np.log2(df_box.aspect.clip(1 / 8, 8)), bins=50, color='#55A868', edgecolor='white')
axes[1].set(title='Tỉ lệ khung log2(w/h)', xlabel='log2(w/h)  (0 = vuông)')
hb = axes[2].hexbin(df_box.w, df_box.h, gridsize=40, bins='log', cmap='viridis', mincnt=1)
axes[2].set(title='w vs h', xlabel='w (px)', ylabel='h (px)'); fig.colorbar(hb, ax=axes[2])
savefig(fig, 'box_sizes')

size_bins = [0, 16, 32, 64, 96, np.inf]
size_labels = ['<16', '16-32', '32-64', '64-96', '>=96']
tbl = pd.DataFrame({
    'ảnh gốc': pd.cut(df_box.side, size_bins, labels=size_labels, right=False).value_counts(normalize=True),
    f'imgsz={YOLO_IMGSZ}': pd.cut(df_box.side_yolo, size_bins, labels=size_labels, right=False).value_counts(normalize=True),
}).reindex(size_labels)
display((tbl * 100).round(1).rename_axis('cạnh (px)').add_suffix(' (%)'))

## 8. Vị trí quả, mật độ và mức độ chồng lấn

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.2))
hb = axes[0].hexbin(df_box.cx_n, df_box.cy_n, gridsize=30, cmap='magma', mincnt=1)
axes[0].invert_yaxis(); axes[0].set(title='Vị trí tâm quả (chuẩn hóa)', xlabel='x / W', ylabel='y / H')
fig.colorbar(hb, ax=axes[0])
axes[1].scatter(df_img['count'], df_img.mean_side, s=12, alpha=0.6, c=df_img.crowded_frac, cmap='coolwarm')
axes[1].set(title='Số quả vs kích thước quả TB', xlabel='số quả / ảnh', ylabel='mean side (px)')
axes[2].hist(df_box.max_iou, bins=50, color='#C44E52', edgecolor='white')
axes[2].axvline(CROWD_IOU, color='k', ls='--', label=f'ngưỡng chồng lấn {CROWD_IOU}')
axes[2].set_yscale('log'); axes[2].set(title='IoU lớn nhất với box khác', xlabel='max IoU'); axes[2].legend()
g = df_img.groupby('count_bin', observed=True)['crowded_frac'].mean()
axes[3].bar(g.index.astype(str), g.values * 100, color='#8172B2')
axes[3].set(title='% quả bị chồng lấn theo nhóm số quả', xlabel='số quả / ảnh', ylabel='% box chồng lấn')
savefig(fig, 'spatial_crowding')

print(f'Box chồng lấn (IoU > {CROWD_IOU}): {(df_box.max_iou > CROWD_IOU).mean():.2%}')
print(f'Box bị phủ ≥ 30% diện tích     : {(df_box.max_ioa >= 0.3).mean():.2%}')
print(f"Tương quan Spearman (số quả, % chồng lấn): {df_img['count'].corr(df_img.crowded_frac, method='spearman'):.3f}")

## 9. Ảnh minh họa theo phân vị số quả

In [ ]:
from src.visualization import draw_boxes, draw_count_banner

srt = df_img.sort_values('count').reset_index(drop=True)
pick = srt.iloc[sorted({int(q * (len(srt) - 1)) for q in (0, 0.25, 0.5, 0.75, 1.0)})]
fig, axes = plt.subplots(1, len(pick), figsize=(5 * len(pick), 4.2), squeeze=False)
for ax, (_, row) in zip(axes[0], pick.iterrows()):
    r = rec_by_id[row.image_id]
    vis = draw_count_banner(draw_boxes(load_image(r.image_path), r.boxes), f'GT: {r.count}')
    ax.imshow(vis); ax.axis('off'); ax.set_title(f'{r.image_id} [{row.split}]', fontsize=9)
savefig(fig, 'samples_by_count')

In [ ]:
# Ảnh đông quả nhất: tô màu instance mask + phóng to vùng dày đặc nhất
r = rec_by_id[srt.iloc[-1].image_id]
img = load_image(r.image_path)
vis = img.copy()
if r.mask_path and Path(r.mask_path).exists():
    from src.data.annotations import read_instance_mask
    inst = read_instance_mask(r.mask_path)
    rng = np.random.default_rng(0)
    out = img.astype(np.float32)
    for i in r.instance_ids:
        m = inst == i
        out[m] = 0.5 * out[m] + 0.5 * rng.integers(60, 255, 3)
    vis = out.astype(np.uint8)
vis = draw_boxes(vis, r.boxes)

pts = np.c_[(r.boxes[:, 0] + r.boxes[:, 2]) / 2, (r.boxes[:, 1] + r.boxes[:, 3]) / 2]
H2, xe, ye = np.histogram2d(pts[:, 0], pts[:, 1], bins=6, range=[[0, r.width], [0, r.height]])
i, j = np.unravel_index(H2.argmax(), H2.shape)
cx, cy, half = (xe[i] + xe[i + 1]) / 2, (ye[j] + ye[j + 1]) / 2, min(r.width, r.height) // 4
x0, y0 = int(np.clip(cx - half, 0, r.width - 2 * half)), int(np.clip(cy - half, 0, r.height - 2 * half))

fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1.6, 1]})
axes[0].imshow(vis); axes[0].add_patch(plt.Rectangle((x0, y0), 2 * half, 2 * half, fill=False, ec='yellow', lw=2))
axes[0].set_title(f'{r.image_id}: {r.count} quả (instance mask + box)'); axes[0].axis('off')
axes[1].imshow(vis[y0:y0 + 2 * half, x0:x0 + 2 * half]); axes[1].set_title('Phóng to vùng dày đặc nhất'); axes[1].axis('off')
savefig(fig, 'densest_image')

## 10. Ảnh gần trùng lặp (rò rỉ giữa các split)
Dùng **dHash 64 bit**: ảnh gần giống nhau có khoảng cách Hamming nhỏ. Cặp gần trùng nằm ở **hai split khác nhau**
làm kết quả test lạc quan giả tạo (ví dụ: các frame liên tiếp của cùng một video / cùng một cây).

In [ ]:
def dhash(path, size=8):
    g = cv2.imread(str(path), cv2.IMREAD_REDUCED_GRAYSCALE_4)
    if g is None:
        return np.zeros(size * size, bool)
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return (g[:, 1:] > g[:, :-1]).flatten()


hashes = np.stack([dhash(r.image_path) for r in tqdm(records, desc='dHash')])
pairs = []
for s in range(0, len(hashes), 512):
    D = (hashes[s:s + 512, None, :] != hashes[None, :, :]).sum(-1)
    for a, b in zip(*np.nonzero(D <= DUP_HAMMING)):
        a += s
        if a < b:
            ia, ib = records[a].image_id, records[b].image_id
            pairs.append({'image_a': ia, 'image_b': ib, 'hamming': int(D[a - s, b]),
                          'split_a': id2split.get(ia), 'split_b': id2split.get(ib)})
df_dup = pd.DataFrame(pairs, columns=['image_a', 'image_b', 'hamming', 'split_a', 'split_b'])
cross = df_dup[df_dup.split_a != df_dup.split_b]
print(f'Cặp gần trùng: {len(df_dup)} | khác split (rò rỉ tiềm năng): {len(cross)}')
display(cross.sort_values('hamming').head(10))

if len(cross):
    show = cross.sort_values('hamming').head(3)
    fig, axes = plt.subplots(len(show), 2, figsize=(10, 3.5 * len(show)), squeeze=False)
    for k, (_, row) in enumerate(show.iterrows()):
        for c, (iid, sp) in enumerate([(row.image_a, row.split_a), (row.image_b, row.split_b)]):
            axes[k][c].imshow(load_image(rec_by_id[iid].image_path)); axes[k][c].axis('off')
            axes[k][c].set_title(f'{iid} [{sp}] (Hamming={row.hamming})', fontsize=9)
    savefig(fig, 'near_duplicates')

## 11. Tổng kết

In [ ]:
cnt = df_img['count']
summary = {
    'dataset': DS_CFG.get('name'), 'synthetic': SYNTHETIC, 'format': DS_CFG.get('format'),
    'n_images': len(records), 'n_fruits': int(cnt.sum()),
    'count_per_image': {k: float(v) for k, v in cnt.describe().items()},
    'splits': {s: {'n_images': int((df_img.split == s).sum()),
                   'mean_count': float(df_img.loc[df_img.split == s, 'count'].mean())} for s in SPLITS},
    'issues': {k: len(v) for k, v in issues.items()},
    'box_checks': {k: int(m.sum()) for k, m in checks.items()},
    'box_side_px': {'median': float(df_box.side.median()), 'p5': float(df_box.side.quantile(0.05)),
                    'p95': float(df_box.side.quantile(0.95))},
    'small_frac_at_yolo_imgsz': float((df_box.side_yolo < 32).mean()),
    'crowded_box_frac': float((df_box.max_iou > CROWD_IOU).mean()),
    'near_duplicate_pairs': len(df_dup), 'cross_split_duplicates': len(cross),
}
if mask_scan is not None:
    summary['mask_scan'] = {'dropped_small': int(mask_scan.n_dropped_small.sum()),
                            'fragmented': int(mask_scan.n_fragmented.sum())}
write_json(summary, OUT_DIR / 'data_audit_summary.json')
df_img.to_csv(OUT_DIR / 'image_stats.csv', index=False)

notes = [
    f"- **{len(records)} ảnh**, **{int(cnt.sum())} quả**; mỗi ảnh {cnt.min():.0f}–{cnt.max():.0f} quả "
    f"(TB {cnt.mean():.1f}, trung vị {cnt.median():.0f}, độ lệch chuẩn {cnt.std():.1f}).",
    f"- Kích thước quả trung vị **{df_box.side.median():.0f}px**; **{(df_box.side_yolo < 32).mean():.0%}** quả nhỏ hơn 32px "
    f"sau khi resize về imgsz={YOLO_IMGSZ}" + (' → nên cân nhắc tiling / tăng imgsz.' if (df_box.side_yolo < 32).mean() > 0.3 else '.'),
    f"- **{(df_box.max_iou > CROWD_IOU).mean():.0%}** box chồng lấn (IoU > {CROWD_IOU}); "
    f"ảnh càng nhiều quả càng chồng lấn (Spearman = {cnt.corr(df_img.crowded_frac, method='spearman'):.2f}).",
    f"- Vấn đề file: " + ', '.join(f'{k.lower()}: {len(v)}' for k, v in issues.items() if len(v)) if any(len(v) for v in issues.values())
    else '- Không phát hiện lỗi file.',
    f"- Ảnh gần trùng khác split: **{len(cross)}** cặp" + (' → cân nhắc chia theo cây / theo video.' if len(cross) else '.'),
]
display(Markdown('### Nhận xét tự động\n' + '\n'.join(notes)))
print('Đã lưu:', OUT_DIR)

### Gợi ý ghi vào báo cáo
- Bảng thống kê số quả theo split (mục 6) + kết quả KS chứng minh split **stratified** cân bằng.
- Biểu đồ kích thước quả (mục 7) giải thích lựa chọn `imgsz = 1024` và chế độ tiling.
- Biểu đồ chồng lấn theo nhóm số quả (mục 8) là cơ sở cho câu hỏi *detection vs density khi quả chồng lấn*.
- Tiếp theo: `02_augmentation.ipynb`.